In [1]:
!pip install transformers datasets -q

In [2]:
import time
from datasets import load_dataset
from transformers import pipeline

print('Loading summarization pipeline...')
summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')
print('Pipeline ready.')

Loading summarization pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Pipeline ready.


## Load a Small Sample (CNN/DailyMail)
Use a tiny subset to keep inference fast.

In [4]:
dataset = load_dataset('cnn_dailymail', '3.0.0', split='test[:5]')
print('Articles loaded:', len(dataset))
print('Example title:', dataset[0]['article'][:120].replace('\n', ' ') + '...')

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Articles loaded: 5
Example title: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a st...


## Generate Summaries

In [6]:
def summarize_batch(texts, max_len=130, min_len=30):
    outputs = []
    start = time.time()
    for txt in texts:
        summary = summarizer(txt, max_length=max_len, min_length=min_len, do_sample=False)[0]['summary_text']
        outputs.append(summary)
    elapsed = time.time() - start
    return outputs, elapsed

articles = dataset['article']
summaries, elapsed = summarize_batch(articles)

for i, (art, summ) in enumerate(zip(articles, summaries), 1):
    print(f"\n=== Sample {i} ===")
    print('Article snippet:', art[:200].replace('\n', ' ') + '...')
    print('Summary:', summ)

print(f"\nTotal time for {len(articles)} summaries: {elapsed:.2f}s")


=== Sample 1 ===
Article snippet: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territor...
Summary:  The Palestinian Authority becomes the 123rd member of the International Criminal Court . The move gives the court jurisdiction over alleged crimes in Palestinian territories . Israel and the United States opposed the Palestinians' efforts to join the court .

=== Sample 2 ===
Article snippet: (CNN)Never mind cats having nine lives. A stray pooch in Washington State has used up at least three of her own after being hit by a car, apparently whacked on the head with a hammer in a misguided me...
Summary:  Theia is a one-year-old bully breed mix who was hit by a car and buried in a field . She was found by a worker who took her to a vet for help . She suffered a dislocated jaw, leg injuries and a caved-in sinus cavity . A fundraising page has ra

## Quick Quality Check (Compression Ratio)
A simple heuristic to see how much shorter the summaries are.

In [7]:
def compression_ratio(src, tgt):
    return len(tgt.split()) / max(1, len(src.split()))

ratios = [compression_ratio(a, s) for a, s in zip(articles, summaries)]
print('Compression ratios:', [f"{r:.2f}" for r in ratios])
print('Average ratio:', sum(ratios) / len(ratios))

Compression ratios: ['0.07', '0.15', '0.06', '0.19', '0.13']
Average ratio: 0.11921690339760249


## Test on Custom Text

In [8]:
custom_article = '''
Transformers have become the dominant architecture for natural language processing tasks.
By relying on self-attention, they capture long-range dependencies efficiently.
This has led to breakthroughs in translation, summarization, and question answering.
'''
custom_summary = summarizer(custom_article, max_length=80, min_length=25, do_sample=False)[0]['summary_text']
print('Custom summary:', custom_summary)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Your max_length is set to 80, but your input_length is only 52. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=26)


Custom summary: Transformers have become the dominant architecture for natural language processing tasks . They capture long-range dependencies efficiently . This has led to breakthroughs in translation, summarization and question answering .


## Notes
- DistilBART is a distilled BART, faster for demos.
- For longer articles, adjust `max_length` and `min_length`.
- On GPU (Colab), generation is much faster; CPU works but slower.